# Customer Churn Analysis and Prediction

This notebook combines exploratory data analysis, feature review, and baseline modeling for a customer churn problem.

## Project Objective

The goal is to understand which customers are more likely to churn and build a machine learning model that predicts churn risk.

## Business Questions

- Which customer groups churn the most?
- Does contract type affect churn?
- Are high monthly charges linked with churn?
- Does long tenure reduce churn?
- Which features seem most important for prediction?

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (8, 4)

BASE_DIR = Path.cwd().parent
DATA_PATH = BASE_DIR / 'data' / 'processed' / 'cleaned_customer_churn.csv'

df = pd.read_csv(DATA_PATH)
df.head()

## Data Overview

In [ ]:
print('Shape:', df.shape)
display(df.head())
display(df.describe(include='all').T.head(10))

In [ ]:
df.info()

In [ ]:
df.isnull().sum().sort_values(ascending=False).head(10)

## Churn Distribution

In [ ]:
churn_rate = df['Churn'].value_counts(normalize=True).mul(100).round(2)
display(churn_rate)

sns.countplot(data=df, x='Churn', palette='Set2')
plt.title('Churn Distribution')
plt.show()

## Contract Type vs Churn

In [ ]:
contract_churn = pd.crosstab(df['Contract'], df['Churn'], normalize='index').round(3)
display(contract_churn)

sns.countplot(data=df, x='Contract', hue='Churn', palette='Set2')
plt.xticks(rotation=15)
plt.title('Contract Type vs Churn')
plt.show()

## Monthly Charges and Tenure

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.boxplot(data=df, x='Churn', y='MonthlyCharges', palette='Set3', ax=axes[0])
axes[0].set_title('Monthly Charges vs Churn')

sns.boxplot(data=df, x='Churn', y='tenure', palette='Pastel1', ax=axes[1])
axes[1].set_title('Tenure vs Churn')

plt.tight_layout()
plt.show()

## Payment Method and Internet Service

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

sns.countplot(data=df, x='PaymentMethod', hue='Churn', palette='Set2', ax=axes[0])
axes[0].tick_params(axis='x', rotation=20)
axes[0].set_title('Payment Method vs Churn')

sns.countplot(data=df, x='InternetService', hue='Churn', palette='Set2', ax=axes[1])
axes[1].set_title('Internet Service vs Churn')

plt.tight_layout()
plt.show()

## Correlation Review

In [ ]:
numeric_df = df.copy()
numeric_df['Churn'] = numeric_df['Churn'].map({'Yes': 1, 'No': 0})
numeric_df = numeric_df.select_dtypes(include=['number'])

corr = numeric_df.corr()
sns.heatmap(corr, cmap='coolwarm', annot=True, fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

## Baseline Modeling

In [ ]:
model_df = df.copy()
if 'customerID' in model_df.columns:
    model_df = model_df.drop(columns=['customerID'])

model_df['Churn'] = model_df['Churn'].map({'Yes': 1, 'No': 0})

X = model_df.drop(columns=['Churn'])
y = model_df['Churn']

categorical_columns = X.select_dtypes(include=['object']).columns.tolist()
numeric_columns = X.select_dtypes(exclude=['object']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
            ]),
            numeric_columns,
        ),
        (
            'cat',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder', OneHotEncoder(handle_unknown='ignore')),
            ]),
            categorical_columns,
        ),
    ]
)

model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000)),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print('ROC-AUC:', round(roc_auc_score(y_test, y_prob), 4))

## Interpretation Notes

Write 3 to 5 short observations here after running the notebook.

- Example: Customers on month-to-month contracts churn more often than customers on long-term contracts.
- Example: Higher monthly charges appear to be associated with higher churn.
- Example: Customers with lower tenure are more likely to leave.

## Final Conclusion

Summarize:

- the most important churn drivers
- how well the baseline model performed
- what the business should do next